In [1]:
import sys

print("Python version:", sys.version)
print("Python location:", sys.executable)

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python location: c:\Users\ausu\Desktop\saudi-tech-research\.venv311\Scripts\python.exe


In [2]:
import json
from pathlib import Path
from urllib.request import urlopen

# Locate the project folder.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

raw_dir = project_dir / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

url = (
    "https://"
    "data.ksu.edu.sa/sites/data.ksu.edu.sa/files/users/user976/"
    "KSU-DMO-OD-DATASET-SciResarch-Publications-2025.json"
)

# Download and check that the content is valid JSON.
with urlopen(url, timeout=90) as response:
    raw_bytes = response.read()

ksu_data = json.loads(raw_bytes.decode("utf-8-sig"))

# Preserve the original downloaded content.
file_path = raw_dir / "ksu_publications_2025.json"
file_path.write_bytes(raw_bytes)

print("Saved to:", file_path)
print("JSON structure:", type(ksu_data).__name__)
print("Preview:")
print(json.dumps(ksu_data, ensure_ascii=False, indent=2)[:2500])

JSONDecodeError: Invalid \escape: line 3368 column 1428 (char 696496)

In [3]:
# Preserve the downloaded bytes exactly as received.
original_path = raw_dir / "ksu_publications_2025_original.json"
original_path.write_bytes(raw_bytes)

text = raw_bytes.decode("utf-8-sig")

try:
    ksu_data = json.loads(text)
except json.JSONDecodeError as error:
    print("Problem:", error.msg)
    print("Line:", error.lineno, "| Column:", error.colno)

    start = max(0, error.pos - 150)
    end = min(len(text), error.pos + 150)

    print("\nText around the problem:")
    print(repr(text[start:end]))

Problem: Invalid \escape
Line: 3368 | Column: 1428

Text around the problem:
'onts} \\\\usepackage{amssymb} \\\\usepackage{amsbsy} \\\\usepackage{mathrsfs} \\\\usepackage{upgreek} \\\\setlength{\\\\oddsidemargin}{-69pt} \\\\begin{document}$$F\\\text {-score}$$\\\\end{document} of 72.78%. While generative performance reached BERT-F\\\\documentclass[12pt]{minimal} \\\\usepackage{amsmath} \\\\usepackag'


In [4]:
try:
    ksu_data = json.loads(text)
except json.JSONDecodeError as error:
    print("Characters around the error:")
    for position in range(max(0, error.pos - 5), error.pos + 12):
        character = text[position]
        print(position, repr(character), f"U+{ord(character):04X}")

Characters around the error:
696491 't' U+0074
696492 '}' U+007D
696493 '$' U+0024
696494 '$' U+0024
696495 'F' U+0046
696496 '\\' U+005C
696497 '\t' U+0009
696498 'e' U+0065
696499 'x' U+0078
696500 't' U+0074
696501 ' ' U+0020
696502 '{' U+007B
696503 '-' U+002D
696504 's' U+0073
696505 'c' U+0063
696506 'o' U+006F
696507 'r' U+0072


In [ ]:
# Create a working copy.
repaired_text = text

position = 696496
fragment = repaired_text[position:position + 2]

# Verify the exact characters before changing anything.
assert fragment == "\\" + "\t", "Characters differ; stop and inspect."



# Encode the backslash and tab correctly for JSON.
replacement = json.dumps(fragment)[1:-1]


repaired_text = (
    repaired_text[:position]
    + replacement
    + repaired_text[position + 2:]
)


try:
    ksu_data = json.loads(repaired_text)
    print("Success: the working copy now loads as JSON.")
    print("JSON structure:", type(ksu_data).__name__)
except json.JSONDecodeError as error:
    print("Another JSON issue:", error.msg)
    print("Position:", error.pos)
    print("Nearby text:", repr(
        repaired_text[max(0, error.pos - 60):error.pos + 60]
    ))

Success: the working copy now loads as JSON.
JSON structure: list


In [6]:
print("Number of records:", len(ksu_data))

if ksu_data:
    first_record = ksu_data[0]

    if isinstance(first_record, dict):
        print("\nField names:")
        for field in first_record:
            print("-", field)

    print("\nFirst record:")
    print(json.dumps(first_record, ensure_ascii=False, indent=2))
else:
    print("The dataset is empty.")

Number of records: 12703

Field names:
- Authors
- Article Title
- Source Title
- Document Type
- Author Keywords
- Abstract
- Affiliations
- DOI

First record:
{
  "Authors": "Al-Baadani, HH; Alharthi, AS; Abbas, NI; Qasem, AA; Saleh, A; Ibraheem, MA",
  "Article Title": "Effect of activated and inactivated Saccharomyces cerevisiae as alternative to antibiotic growth promoter on the performance and health of broilers infected with Clostridium perfringens",
  "Source Title": "ITALIAN JOURNAL OF ANIMAL SCIENCE",
  "Document Type": "Article",
  "Author Keywords": "S. cerevisiae; necrotic enteritis; performance; health; broilers",
  "Abstract": "The study aims to evaluate the addition of activated and inactivated Saccharomyces cerevisiae on the performance and health of broilers infected with Clostridium perfringens. 360 1-day-old male Ross 308 broilers were divided into 5 groups of 12 cages as experimental units (6 birds per cage) as follows: NC = negative control, basal diet; PC = posit

In [7]:
# This is individual publication data, and the file contains 12,703 records


# Create a folder for working data, separate from the original raw files.
interim_dir = project_dir / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)

# Choose a filename for the repaired JSON.
working_path = interim_dir / "ksu_publications_2025_repaired.json"

# Save the repaired text without changing the original downloaded file.
# UTF-8 preserves Arabic and English characters.
working_path.write_text(repaired_text, encoding="utf-8")

# Document the source and exactly what we repaired.
repair_note = {
    # The address used to download the original file.
    "source_url": url,

    # This is the year on the source file, not a verified year per paper.
    "source_file_year": 2025,

    # Count the records in the parsed JSON list.
    "record_count": len(ksu_data),

    # Store paths relative to the project so teammates can use them.
    "original_file": original_path.relative_to(project_dir).as_posix(),
    "working_file": working_path.relative_to(project_dir).as_posix(),

    # Record the zero-based character position of the repair.
    "repair_position": position,

    # Explain the change and its limits.
    "repair": (
        "Escaped a backslash followed by a literal tab to make valid JSON. "
        "Preserved both character values; did not reconstruct LaTeX."
    ),
}

# Save the repair note as a separate, readable JSON file.
note_path = interim_dir / "ksu_2025_repair_note.json"
note_path.write_text(
    json.dumps(repair_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Confirm which files were saved.
print("Saved working copy:", working_path.name)
print("Saved repair note:", note_path.name)

Saved working copy: ksu_publications_2025_repaired.json
Saved repair note: ksu_2025_repair_note.json


In [8]:
# Use separate variables so the 2025 data stays available.
url_2024 = (
    "https://"
    "data.ksu.edu.sa/sites/data.ksu.edu.sa/files/users/user976/"
    "KSU-DMO-OD-DATASET-SciResarch-Publications-2024.json"
)

# Download the file with a 90-second timeout.
with urlopen(url_2024, timeout=90) as response:
    raw_bytes_2024 = response.read()

# Save the original bytes before trying to parse the JSON.
original_path_2024 = raw_dir / "ksu_publications_2024_original.json"
original_path_2024.write_bytes(raw_bytes_2024)

print("Original saved:", original_path_2024.name)

# Decode the text and check whether it is valid JSON.
text_2024 = raw_bytes_2024.decode("utf-8-sig")

try:
    ksu_data_2024 = json.loads(text_2024)
    print("JSON structure:", type(ksu_data_2024).__name__)

    if isinstance(ksu_data_2024, list):
        print("Number of records:", len(ksu_data_2024))

        if ksu_data_2024:
            print("\nFirst record:")
            print(json.dumps(
                ksu_data_2024[0], ensure_ascii=False, indent=2
            ))
    else:
        print("Preview:", repr(ksu_data_2024)[:1500])

except json.JSONDecodeError as error:
    # Report any issue without changing the downloaded content.
    print("JSON issue:", error.msg)
    print("Line:", error.lineno, "| Column:", error.colno)
    print("Position:", error.pos)
    print("Nearby text:", repr(
        text_2024[max(0, error.pos - 100):error.pos + 100]
    ))

Original saved: ksu_publications_2024_original.json
JSON structure: list
Number of records: 16134

First record:
{
  "Authors": "Verma, M; Meena, RK; Khan, MI; Khatib, JM",
  "Article Title": "Analysis of the effect of curing and mixing periods on mechanical properties of the geopolymer composite",
  "Source Title": "MATERIALS SCIENCE-POLAND",
  "Document Type": "Article",
  "Author Keywords": "Geopolymer concrete; Alkali-silica reaction; Sustainable Construction; Durability; Heat curing",
  "Affiliations": "GLA University; National Institute of Technology (NIT System); National Institute of Technology Delhi; King Saud University; University of Wolverhampton"
}


In [9]:
# Keep the 2023 data separate from the other years.
url_2023 = (
    "https://"
    "data.ksu.edu.sa/sites/data.ksu.edu.sa/files/users/user976/"
    "KSU-DMO-OD-DATASET-SciResarch-Publications-2023.json"
)

# Download the original file.
with urlopen(url_2023, timeout=90) as response:
    raw_bytes_2023 = response.read()

# Preserve the downloaded bytes unchanged.
original_path_2023 = raw_dir / "ksu_publications_2023_original.json"
original_path_2023.write_bytes(raw_bytes_2023)

print("Original saved:", original_path_2023.name)

# Check whether the file contains valid JSON.
text_2023 = raw_bytes_2023.decode("utf-8-sig")

try:
    ksu_data_2023 = json.loads(text_2023)
    print("JSON structure:", type(ksu_data_2023).__name__)

    if isinstance(ksu_data_2023, list):
        print("Number of records:", len(ksu_data_2023))

        if ksu_data_2023:
            print("\nFirst record:")
            print(json.dumps(
                ksu_data_2023[0], ensure_ascii=False, indent=2
            ))
    else:
        print("Preview:", repr(ksu_data_2023)[:1500])

except json.JSONDecodeError as error:
    # Report problems without modifying the original file.
    print("JSON issue:", error.msg)
    print("Line:", error.lineno, "| Column:", error.colno)
    print("Position:", error.pos)
    print("Nearby text:", repr(
        text_2023[max(0, error.pos - 100):error.pos + 100]
    ))

Original saved: ksu_publications_2023_original.json
JSON structure: list
Number of records: 12346

First record:
{
  "Authors": "Abbas, H; Tao, W; Khan, G; Alrefaei, AF; Iqbal, J; Albeshr, MF; Kulsoom, I",
  "Article Title": "Multilayer perceptron and Markov Chain analysis based hybrid-approach for predicting land use land cover change dynamics with Sentinel-2 imagery",
  "Source Title": "GEOCARTO INTERNATIONAL",
  "Document Type": "Article",
  "Author Keywords": "Land use land cover; future prediction; multilayer perceptron; Markov Chain analysis; Sentinel 2",
  "Abstract": "As urbanization accelerates, the degree of human impact on land use is increasing. land use land cover change (LULC) is acknowledged as crucial factor in environmental change. The best way to understand historical land use patterns, changes, drivers, and developments is through a rigorous assessment of LULC changes. In this study, we aim to identify LULC changes from 2015 to 2022, and predict changes for 2030. Sen

In [10]:
# Use the parsed datasets; the 2025 data uses our repaired working copy.
datasets = {
    2023: ksu_data_2023,
    2024: ksu_data_2024,
    2025: ksu_data,
}

availability_report = []

for year, records in datasets.items():
    # Stop if any record has an unexpected structure.
    assert all(isinstance(record, dict) for record in records)

    print(f"\n--- Source file year: {year} ---")
    print("Total records:", len(records))

    for field in ["DOI", "Abstract"]:
        # Count absent fields separately from fields with empty values.
        absent = sum(field not in record for record in records)

        empty = sum(
            field in record
            and (
                record[field] is None
                or (
                    isinstance(record[field], str)
                    and not record[field].strip()
                )
            )
            for record in records
        )

        populated = len(records) - absent - empty

        availability_report.append({
            "source_file_year": year,
            "field": field,
            "total_records": len(records),
            "absent": absent,
            "empty": empty,
            "populated": populated,
        })

        print(
            f"{field}: {populated:,} populated | "
            f"{absent:,} absent | {empty:,} empty"
        )

# Save the findings for your source notes and enrichment planning.
report_path = interim_dir / "ksu_field_availability.json"
report_path.write_text(
    json.dumps(availability_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nReport saved:", report_path.name)


--- Source file year: 2023 ---
Total records: 12346
DOI: 2,986 populated | 0 absent | 9,360 empty
Abstract: 12,339 populated | 0 absent | 7 empty

--- Source file year: 2024 ---
Total records: 16134
DOI: 0 populated | 16,134 absent | 0 empty
Abstract: 0 populated | 16,134 absent | 0 empty

--- Source file year: 2025 ---
Total records: 12703
DOI: 12,703 populated | 0 absent | 0 empty
Abstract: 12,703 populated | 0 absent | 0 empty

Report saved: ksu_field_availability.json


In [11]:

#make an initial technology filter using titles and keywords, 
# which are available across all three years.
#  This first pass includes research applying technology in other fields, such as AI in medicine or agriculture.

import re

# Starting vocabulary: we can expand it after reviewing the results.
tech_terms = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "neural network",
    "neural networks",
    "multilayer perceptron",
    "random forest",
    "computer vision",
    "natural language processing",
    "large language model",
    "large language models",
    "generative ai",
    "cybersecurity",
    "cyber security",
    "information security",
    "intrusion detection",
    "cryptography",
    "blockchain",
    "software engineering",
    "internet of things",
    "iot",
    "cloud computing",
    "edge computing",
    "wireless network",
    "wireless networks",
    "robotics",
    "robot",
    "robots",
    "data mining",
    "big data",
    "computer science",
]

# Match complete phrases, ignoring capitalization.
patterns = {
    term: re.compile(r"\b" + re.escape(term) + r"\b", re.IGNORECASE)
    for term in tech_terms
}

tech_candidates = {}

for year, records in datasets.items():
    selected = []

    for row_index, record in enumerate(records):
        # Use the same fields for every year.
        searchable_text = " ".join(
            str(record.get(field) or "")
            for field in ["Article Title", "Author Keywords"]
        )

        matched_terms = [
            term
            for term, pattern in patterns.items()
            if pattern.search(searchable_text)
        ]

        if matched_terms:
            # Copy the record so the original dataset remains unchanged.
            candidate = record.copy()
            candidate["source_file_year"] = year
            candidate["source_row_index"] = row_index
            candidate["tech_matched_terms"] = matched_terms
            selected.append(candidate)

    tech_candidates[year] = selected

    print(f"\n{year}: {len(selected):,} candidates / {len(records):,} records")

    # Show a few titles so we can review the filter.
    for candidate in selected[:5]:
        print("-", candidate.get("Article Title"))
        print("  Matched:", ", ".join(candidate["tech_matched_terms"]))


2023: 938 candidates / 12,346 records
- Multilayer perceptron and Markov Chain analysis based hybrid-approach for predicting land use land cover change dynamics with Sentinel-2 imagery
  Matched: multilayer perceptron
- Enhancing deep learning techniques for the diagnosis of the novel coronavirus (COVID-19) using X-ray images
  Matched: artificial intelligence, deep learning, neural networks
- A Review of the Scope, Future, and Effectiveness of Using Artificial Intelligence in Cardiac Rehabilitation: A Call to Action for the Kingdom of Saudi Arabia
  Matched: artificial intelligence
- Modelling of land use and land cover changes and prediction using CA-Markov and Random Forest
  Matched: random forest
- County-level corn yield prediction using supervised machine learning
  Matched: machine learning

2024: 1,306 candidates / 16,134 records
- 3-D Trajectory Optimization and Communication Resources Allocation in UAV-Assisted IoT Networks for Sustainable Industry 5.0
  Matched: iot
- 6G-E

In [12]:

# The filter selected 3,533 candidate records across the three files. 
# The examples fit our broad scope, including technology applied to healthcare and agriculture. 
# These counts are before deduplication, and we still need to review some matches and excluded records.


# Save each year's candidates separately.
# Original downloaded files remain unchanged.
for year, candidates in tech_candidates.items():
    output_path = interim_dir / f"ksu_tech_candidates_{year}.json"

    output_path.write_text(
        json.dumps(candidates, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"Saved {len(candidates):,} candidates: {output_path.name}")

# Document how the initial filter selected records.
filter_note = {
    "status": "Initial candidates; manual review pending",
    "scope": "Technology research, including applications in other fields",
    "fields_searched": ["Article Title", "Author Keywords"],
    "matching_method": "Case-insensitive phrases with word boundaries",
    "terms": tech_terms,
    "counts_by_source_file_year": {
        str(year): len(candidates)
        for year, candidates in tech_candidates.items()
    },
    "limitations": [
        "Keyword matching may include irrelevant or miss relevant papers.",
        "Source file year is not a verified publication year.",
        "Records have not yet been deduplicated.",
    ],
    "enrichment_reminder": (
        "All 2024 source records lack DOI and Abstract fields. "
        "2023 also has missing values. Recheck missing fields among "
        "selected candidates before API enrichment."
    ),
}

# Save the filter documentation alongside the candidate files.
note_path = interim_dir / "ksu_tech_filter_note.json"
note_path.write_text(
    json.dumps(filter_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved filter documentation:", note_path.name)

Saved 938 candidates: ksu_tech_candidates_2023.json
Saved 1,306 candidates: ksu_tech_candidates_2024.json
Saved 1,289 candidates: ksu_tech_candidates_2025.json
Saved filter documentation: ksu_tech_filter_note.json


In [13]:
import random

# A fixed seed makes the sample reproducible.
rng = random.Random(42)

for year, candidates in tech_candidates.items():
    print(f"\n--- {year}: selected papers for review ---")

    sample = rng.sample(candidates, min(5, len(candidates)))

    for number, record in enumerate(sample, start=1):
        print(f"\n{number}. {record.get('Article Title', '')}")
        print("Keywords:", record.get("Author Keywords", ""))
        print("Matched terms:", ", ".join(record["tech_matched_terms"]))
        print("Original row index:", record["source_row_index"])


--- 2023: selected papers for review ---

1. A Novel Hybrid MPPT Approach for Solar PV Systems Using Particle-Swarm-Optimization-Trained Machine Learning and Flying Squirrel Search Optimization
Keywords: DC-DC converter; MPPT algorithm; solar photovoltaic system
Matched terms: machine learning
Original row index: 10091

2. Smart Deep Learning Model to Recognize PCM Optimization Performance on Solar Cooling System
Keywords: PCM optimization; photovoltaic systems; renewable energy
Matched terms: deep learning
Original row index: 1856

3. Solid oxide fuel cell energy system with absorption-ejection refrigeration optimized using a neural network with multiple objectives
Keywords: Solid oxide fuel cell; Absorption-ejection refrigeration; Neural network Genetic algorithm Energy efficiency
Matched terms: neural network
Original row index: 381

4. Optimal Fuzzy Wavelet Neural Network Based Road Damage Detection
Keywords: Flooding; road damage; machine learning; parameter tuning; computer visi

In [19]:

# Rebuild the candidate lists with a more precise matching rule.
tech_candidates = {}

for year, records in datasets.items():
    selected = []

    for row_index, record in enumerate(records):
        searchable_text = " ".join(
            str(record.get(field) or "")
            for field in ["Article Title", "Author Keywords"]
        )

        # Avoid confusing the eye-health condition with computing research.
        text_for_matching = re.sub(
            r"\bcomputer\s+vision\s+syndrome\b",
            " ",
            searchable_text,
            flags=re.IGNORECASE,
        )

        matched_terms = [
            term
            for term, pattern in patterns.items()
            if pattern.search(text_for_matching)
        ]

        if matched_terms:
            candidate = record.copy()
            candidate["source_file_year"] = year
            candidate["source_row_index"] = row_index
            candidate["tech_matched_terms"] = matched_terms
            selected.append(candidate)

    tech_candidates[year] = selected
    print(f"{year}: {len(selected):,} candidates")

# Update the saved candidate files.
for year, candidates in tech_candidates.items():
    output_path = interim_dir / f"ksu_tech_candidates_{year}.json"
    output_path.write_text(
        json.dumps(candidates, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

# Document the revised rule and counts.
filter_note["excluded_phrase_before_matching"] = "computer vision syndrome"
filter_note["counts_by_source_file_year"] = {
    str(year): len(candidates)
    for year, candidates in tech_candidates.items()
}

(interim_dir / "ksu_tech_filter_note.json").write_text(
    json.dumps(filter_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Updated candidate files and filter notes.")

2023: 940 candidates
2024: 1,306 candidates
2025: 1,289 candidates
Updated candidate files and filter notes.


In [15]:
# Use a fixed seed so this review sample is reproducible.
rng = random.Random(42)

for year, records in datasets.items():
    # Identify the original rows already selected by the filter.
    selected_indices = {
        record["source_row_index"]
        for record in tech_candidates[year]
    }

    excluded = [
        (index, record)
        for index, record in enumerate(records)
        if index not in selected_indices
    ]

    print(f"\n--- {year}: excluded papers for review ---")

    for index, record in rng.sample(excluded, min(5, len(excluded))):
        print("\nOriginal row index:", index)
        print("Title:", record.get("Article Title", ""))
        print("Keywords:", record.get("Author Keywords", ""))


--- 2023: excluded papers for review ---

Original row index: 11212
Title: Estimation of electrostatic and covalent contributions to the enthalpy of H-bond formation in H-complexes of 1,2,3-benzotriazole with proton-acceptor molecules by IR spectroscopy and DFT calculations
Keywords: H-bond; 1,2,3-benzotriazole; DFT; AIM; ELF; NCI; RDG

Original row index: 1945
Title: Cu2ZnGeSe4 single crystals: Growth, structure and temperature dependence of band gap
Keywords: Single crystal growth; Crystal structure; Band gap; Gas chemical method; Semiconducting quaternary alloys

Original row index: 439
Title: Impact of varied fog collector designs on fog and rainwater harvesting under fluctuating wind speed and direction
Keywords: Fog; Rainwater; Mesh design; ANN modeling; Arid region

Original row index: 4817
Title: A review on recent advances in covalent organic frameworks-based membranes: Synthesis, modification, and applications in liquid phase separation
Keywords: Covalent-organic frameworks;

In [16]:

#found a likely missed match: the 2023 fog-collector paper lists 
# “ANN modeling”, commonly used for artificial neural network modeling. 
# Let’s inspect its abstract before adding that abbreviation to the filter.


# Next: inspect the ANN paper.

# Inspect the original record without changing it.
record = datasets[2023][439]

print("Title:", record.get("Article Title", ""))
print("\nKeywords:", record.get("Author Keywords", ""))
print("\nAbstract:", record.get("Abstract") or "No abstract available")

Title: Impact of varied fog collector designs on fog and rainwater harvesting under fluctuating wind speed and direction

Keywords: Fog; Rainwater; Mesh design; ANN modeling; Arid region

Abstract: This study explores innovative approaches to fog water collection by introducing a novel cylindrical mesh design, deviating from the traditional flat mesh. Motivated by the need to address the impact of variable wind directions on collection efficiency, field and laboratory experiments were conducted. Field experiments incorporated both square and cylindrical mesh designs, strategically placed to accommodate natural wind variations. Simultaneously, laboratory experiments considered two wind speeds (7 and 9 km/h) for both designs. Results indicated higher water collection quantities at 7 km/h compared to 9 km/h for both designs. The study emphasizes the critical role of wind direction stability. In regions with consistent wind directions during fog and rain events, the square design proves mo

In [17]:

#Confirmed: this paper uses artificial neural networks, so it fits our scope of computing and AI applications.
#Next, add “ANN modeling” and “ANN modelling” to the filter. 
# These phrases are more specific than matching ANN alone.


# Add both spellings without creating duplicate entries.
for term in ["ann modeling", "ann modelling"]:
    if term not in tech_terms:
        tech_terms.append(term)

# Rebuild the matching patterns with the updated vocabulary.
patterns = {
    term: re.compile(r"\b" + re.escape(term) + r"\b", re.IGNORECASE)
    for term in tech_terms
}

# Keep the saved documentation aligned with the updated vocabulary.
filter_note["terms"] = tech_terms.copy()
filter_note["review_finding"] = (
    "The 2023 record at row index 439 was missed. Its abstract confirms "
    "artificial neural networks, and its keywords contain ANN modeling. "
    "Added ANN modeling and ANN modelling to the vocabulary."
)

print("Added ANN modeling and ANN modelling.")

Added ANN modeling and ANN modelling.


In [20]:

candidate_availability = []

for year, candidates in tech_candidates.items():
    print(f"\n--- {year}: {len(candidates):,} tech candidates ---")

    for field in ["DOI", "Abstract"]:
        # Count absent, null, or whitespace-only values as missing.
        missing = sum(
            record.get(field) is None
            or (
                isinstance(record.get(field), str)
                and not record[field].strip()
            )
            for record in candidates
        )

        candidate_availability.append({
            "source_file_year": year,
            "field": field,
            "candidate_count": len(candidates),
            "missing_count": missing,
        })

        print(f"{field}: {missing:,} missing")

# Save the findings for API enrichment planning.
report_path = interim_dir / "ksu_tech_enrichment_needs.json"
report_path.write_text(
    json.dumps(candidate_availability, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nSaved:", report_path.name)


--- 2023: 940 tech candidates ---
DOI: 748 missing
Abstract: 0 missing

--- 2024: 1,306 tech candidates ---
DOI: 1,306 missing
Abstract: 1,306 missing

--- 2025: 1,289 tech candidates ---
DOI: 0 missing
Abstract: 0 missing

Saved: ksu_tech_enrichment_needs.json


In [ ]:
enrichment_queue = []

for year, candidates in tech_candidates.items():
    for record in candidates:
        # Identify which fields need enrichment.
        missing_fields = [
            field
            for field in ["DOI", "Abstract"]
            if record.get(field) is None
            or (
                isinstance(record.get(field), str)
                and not record[field].strip()
            )
        ]

        if missing_fields:
            enrichment_queue.append({
                "source_file_year": year,
                "source_row_index": record["source_row_index"],
                "title": record.get("Article Title", ""),
                "authors": record.get("Authors", ""),
                "journal": record.get("Source Title", ""),
                "existing_doi": record.get("DOI"),
                "missing_fields": missing_fields,
                "status": "pending",
            })

# Save lookup requests without modifying the candidate datasets.
queue_path = interim_dir / "ksu_enrichment_queue.json"
queue_path.write_text(
    json.dumps(enrichment_queue, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Records queued:", len(enrichment_queue))
print("Saved:", queue_path.name)


Records queued: 2054
Saved: ksu_enrichment_queue.json


## API lookup sample and match review



In [22]:
import json
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlencode
from urllib.request import Request, urlopen

# Load the saved queue so this section can run after a kernel restart.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

interim_dir = project_dir / "data" / "interim"
queue = json.loads(
    (interim_dir / "ksu_enrichment_queue.json").read_text(encoding="utf-8")
)

# Start with one 2024 paper, where both DOI and abstract are missing.
paper = next(item for item in queue if item["source_file_year"] == 2024)

# Search using title, authors, and journal.
# Do not filter by year: the source file year is not a verified paper year.
query = " ".join(
    str(paper.get(field) or "")
    for field in ["title", "authors", "journal"]
)

lookup_url = (
    "https://" + "api.crossref.org/works?"
    + urlencode({"query.bibliographic": query, "rows": 3})
)

request = Request(
    lookup_url,
    headers={"User-Agent": "KSUResearchStudentProject/0.1"},
)

# Save the API response unchanged, separately from our review notes.
api_dir = project_dir / "data" / "raw" / "crossref"
api_dir.mkdir(parents=True, exist_ok=True)

with urlopen(request, timeout=60) as response:
    api_bytes = response.read()

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
response_path = api_dir / f"lookup_sample_{timestamp}.json"
response_path.write_bytes(api_bytes)

matches = json.loads(api_bytes)["message"]["items"]

# Record which source paper produced this API request.
lookup_note = {
    "source_record": paper,
    "request_url": lookup_url,
    "retrieved_at": datetime.now(timezone.utc).isoformat(),
    "response_file": response_path.relative_to(project_dir).as_posix(),
    "status": "pending_manual_review",
}

(interim_dir / f"lookup_review_{timestamp}.json").write_text(
    json.dumps(lookup_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("ORIGINAL KSU PAPER")
print("Title:", paper["title"])
print("Authors:", paper["authors"])
print("Journal:", paper["journal"])

# Display candidates for comparison; do not assign metadata yet.
for number, match in enumerate(matches, start=1):
    authors = "; ".join(
        " ".join(filter(None, [author.get("given"), author.get("family")]))
        or author.get("name", "")
        for author in match.get("author", [])
    )

    print(f"\n--- Candidate {number} ---")
    print("Title:", "; ".join(match.get("title", [])))
    print("Authors:", authors)
    print("Journal:", "; ".join(match.get("container-title", [])))
    print("Published:", match.get("published", {}).get("date-parts"))
    print("DOI:", match.get("DOI"))
    print("Abstract available:", bool(match.get("abstract")))

ORIGINAL KSU PAPER
Title: 3-D Trajectory Optimization and Communication Resources Allocation in UAV-Assisted IoT Networks for Sustainable Industry 5.0
Authors: Du, PF; Shi, YQ; Cao, HT; Garg, S; Kaddoum, G; Alrashoud, M
Journal: IEEE TRANSACTIONS ON CONSUMER ELECTRONICS

--- Candidate 1 ---
Title: 3-D Trajectory Optimization and Communication Resources Allocation in UAV-Assisted IoT Networks for Sustainable Industry 5.0
Authors: Pengfei Du; Yueqiang Shi; Haotong Cao; Sahil Garg; Georges Kaddoum; Mubarak Alrashoud
Journal: IEEE Transactions on Consumer Electronics
Published: [[2024, 2]]
DOI: 10.1109/tce.2023.3325131
Abstract available: False

--- Candidate 2 ---
Title: Softwarized Resource Allocation in Digital Twins-Empowered Networks for Future Quantum-Enabled Consumer Applications
Authors: Haotong Cao; Sahil Garg; Shahid Mumtaz; Mubarak Alrashoud; Longxiang Yang; Georges Kaddoum
Journal: IEEE Transactions on Consumer Electronics
Published: [[2024, 2]]
DOI: 10.1109/tce.2024.3370052
Ab

In [ ]:
# Candidate 1 is a strong match: 
# its title matches exactly, all six authors align with the KSU names and initials, and the journal matches. 


# Crossref reports February 2024 as the publication date; the 2023 inside the DOI isn’t the publication year.


# Candidates 2 and 3 have different titles,  so we’ll reject them. 
# Candidate 1 has no abstract, so our next lookup will be OpenAlex using its DOI.


In [23]:
# Select the candidate we reviewed.
confirmed_match = matches[0]

# Check that this is still the expected result before saving.
assert confirmed_match.get("DOI", "").lower() == (
    "10.1109/tce.2023.3325131"
)

# Keep the decision separate from the original KSU records.
reviewed_match = {
    "source_file_year": paper["source_file_year"],
    "source_row_index": paper["source_row_index"],
    "source_title": paper["title"],
    "confirmed_doi": confirmed_match["DOI"].lower(),
    "metadata_source": "Crossref",
    "match_status": "reviewed_match",
    "match_reason": (
        "Exact title; all six authors align by surname and initials; "
        "journal matches."
    ),
    "crossref_published": confirmed_match.get("published"),
    "crossref_response_file": lookup_note["response_file"],
    "abstract_status": "missing; OpenAlex lookup pending",
}

review_path = interim_dir / "ksu_sample_confirmed_match.json"
review_path.write_text(
    json.dumps(reviewed_match, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved reviewed DOI:", reviewed_match["confirmed_doi"])
print("Next: look up this DOI in OpenAlex for an abstract.")

Saved reviewed DOI: 10.1109/tce.2023.3325131
Next: look up this DOI in OpenAlex for an abstract.


In [24]:
# Look up the exact DOI we verified through Crossref.
doi_to_lookup = reviewed_match["confirmed_doi"]

openalex_url = (
    "https://" + "api.openalex.org/works/https://doi.org/"
    + doi_to_lookup
)

request = Request(
    openalex_url,
    headers={"User-Agent": "KSUResearchStudentProject/0.1"},
)

# Download the OpenAlex record.
with urlopen(request, timeout=60) as response:
    openalex_bytes = response.read()

# Save the original API response unchanged.
openalex_dir = project_dir / "data" / "raw" / "openalex"
openalex_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
openalex_path = openalex_dir / f"sample_lookup_{stamp}.json"
openalex_path.write_bytes(openalex_bytes)

openalex_work = json.loads(openalex_bytes)

# Display identifying information for us to review.
print("Title:", openalex_work.get("display_name"))
print("DOI:", openalex_work.get("doi"))
print("Publication date:", openalex_work.get("publication_date"))
print("OpenAlex ID:", openalex_work.get("id"))

# OpenAlex stores available abstracts as words with position numbers.
abstract_index = openalex_work.get("abstract_inverted_index")

if abstract_index:
    positioned_words = [
        (position, word)
        for word, positions in abstract_index.items()
        for position in positions
    ]

    openalex_abstract = " ".join(
        word for position, word in sorted(positioned_words)
    )

    print("\nAbstract:")
    print(openalex_abstract)
else:
    openalex_abstract = None
    print("\nNo abstract is available in this OpenAlex record.")

print("\nSaved API response:", openalex_path.name)

Title: 3-D Trajectory Optimization and Communication Resources Allocation in UAV-Assisted IoT Networks for Sustainable Industry 5.0
DOI: https://doi.org/10.1109/tce.2023.3325131
Publication date: 2023-10-23
OpenAlex ID: https://openalex.org/W4387885598

Abstract:
Unmanned aerial vehicle (UAV) has been utilized as an efficient data collector for Internet of Things (IoT) networks in sustainable industry 5.0. Whereas, how to sustain a stable power for the energy-constrained IoT devices (IoTDs) and to enhance the data gathering throughput of UAV-aided IoT networks via the wireless power transfer (WPT) or non-orthogonal multiple access (NOMA) is a twofold challenge. Thus, we propose to maximize the minimum UAV data collection throughput from the IoTDs via jointly optimizing the three-dimensional (3D) trajectories of two UAVs, scheduling and transmitting power of the IoTDs subject to the maximum flight velocity and minimum safe distance for the UAVs, as well as the harvested energy causality

In [25]:

# Confirm that OpenAlex returned the DOI we requested.
returned_doi = (openalex_work.get("doi") or "").lower()
expected_doi = reviewed_match["confirmed_doi"].lower()

assert returned_doi.removeprefix("https://doi.org/") == expected_doi, (
    "DOI mismatch: stop and review."
)

# Record the added metadata and where each value came from.
sample_enrichment = {
    "source_file_year": reviewed_match["source_file_year"],
    "source_row_index": reviewed_match["source_row_index"],
    "source_title": reviewed_match["source_title"],

    "confirmed_doi": expected_doi,
    "doi_source": "Crossref",
    "doi_match_reason": reviewed_match["match_reason"],

    "abstract": openalex_abstract,
    "abstract_source": "OpenAlex",
    "openalex_id": openalex_work.get("id"),

    # Preserve both source dates without choosing a final year yet.
    "crossref_published": reviewed_match["crossref_published"],
    "openalex_publication_date": openalex_work.get("publication_date"),
    "quality_flags": ["publication_date_difference_needs_review"],

    # Link this result to the saved API evidence.
    "crossref_response_file": reviewed_match["crossref_response_file"],
    "openalex_response_file": openalex_path.relative_to(
        project_dir
    ).as_posix(),
}

# Save separately; do not overwrite the original KSU records.
sample_path = interim_dir / "ksu_sample_enrichment.json"
sample_path.write_text(
    json.dumps(sample_enrichment, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved:", sample_path.name)
print("DOI and abstract recorded with their sources.")
print("Publication date difference flagged for review.")


Saved: ksu_sample_enrichment.json
DOI and abstract recorded with their sources.
Publication date difference flagged for review.


In [ ]:
# Identify the paper we already reviewed.
completed_key = (
    reviewed_match["source_file_year"],
    reviewed_match["source_row_index"],
)

# Select two more papers from each year needing enrichment.
# This step selects records only; it does not call an API.
next_sample = []

for year in [2023, 2024]:
    eligible = [
        item
        for item in queue
        if item["source_file_year"] == year
        and (
            item["source_file_year"],
            item["source_row_index"],
        ) != completed_key
    ]

    next_sample.extend(eligible[:2])

# Save the sample so we can reproduce this trial.
sample_queue_path = interim_dir / "ksu_next_lookup_sample.json"
sample_queue_path.write_text(
    json.dumps(next_sample, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Show which papers we will look up.
for number, item in enumerate(next_sample, start=1):
    print(f"\n{number}. Source file year: {item['source_file_year']}")
    print("Title:", item["title"])
    print("Missing fields:", ", ".join(item["missing_fields"]))

print("\nSaved:", sample_queue_path.name)


# retrieve possible 
# Crossref matches for these four papers.


1. Source file year: 2023
Title: Non-Linear Financial Intelligence Mixed-Integer Framework for Strategic Planning in Cost Management
Missing fields: DOI

2. Source file year: 2023
Title: Empowering smart cities: High-altitude platforms based Mobile Edge Computing and Wireless Power Transfer for efficient IoT data processing
Missing fields: DOI

3. Source file year: 2024
Title: 6G-Enabled Consumer Electronics Device Intrusion Detection With Federated Meta-Learning and Digital Twins in a Meta-Verse Environment
Missing fields: DOI, Abstract

4. Source file year: 2024
Title: 6GTelMED: Resources Recommendation Framework on 6G-Enabled Distributed Telemedicine Using Edge-AI
Missing fields: DOI, Abstract

Saved: ksu_next_lookup_sample.json


In [27]:
# Now retrieve three possible Crossref matches for each of 
# these four papers. 
# We’ll review the results before accepting any DOI.


import time
from urllib.error import HTTPError, URLError

# Keep these results separate from the first sample's variables.
batch_reviews = []

crossref_dir = project_dir / "data" / "raw" / "crossref"
crossref_dir.mkdir(parents=True, exist_ok=True)

batch_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

for number, source_paper in enumerate(next_sample, start=1):
    print(f"\n{'=' * 60}")
    print(f"PAPER {number}")
    print("KSU title:", source_paper["title"])
    print("KSU authors:", source_paper["authors"])
    print("KSU journal:", source_paper["journal"])

    # Search using the available bibliographic details.
    query = " ".join(
        str(source_paper.get(field) or "")
        for field in ["title", "authors", "journal"]
    )

    request_url = (
        "https://" + "api.crossref.org/works?"
        + urlencode({"query.bibliographic": query, "rows": 3})
    )

    review = {
        "source_record": source_paper,
        "request_url": request_url,
        "status": "pending",
    }

    try:
        request = Request(
            request_url,
            headers={"User-Agent": "KSUResearchStudentProject/0.1"},
        )

        with urlopen(request, timeout=60) as response:
            response_bytes = response.read()

        # Preserve each API response unchanged.
        response_file = (
            crossref_dir / f"sample_{batch_stamp}_{number}.json"
        )
        response_file.write_bytes(response_bytes)

        candidates = json.loads(response_bytes)["message"]["items"]

        review.update({
            "status": (
                "pending_manual_review" if candidates else "no_results"
            ),
            "retrieved_at": datetime.now(timezone.utc).isoformat(),
            "response_file": response_file.relative_to(
                project_dir
            ).as_posix(),
            "candidates": candidates,
        })

        for rank, candidate in enumerate(candidates, start=1):
            author_names = "; ".join(
                " ".join(filter(None, [
                    author.get("given"),
                    author.get("family"),
                ])) or author.get("name", "")
                for author in candidate.get("author", [])
            )

            print(f"\nCandidate {rank}")
            print("Title:", "; ".join(candidate.get("title", [])))
            print("Authors:", author_names)
            print("Journal:", "; ".join(
                candidate.get("container-title", [])
            ))
            print("DOI:", candidate.get("DOI"))
            print("Published:", candidate.get(
                "published", {}
            ).get("date-parts"))
            print("Abstract available:", bool(candidate.get("abstract")))

    except HTTPError as error:
        review["status"] = "request_failed"
        review["error"] = f"HTTP {error.code}"
        print("Request failed:", review["error"])

        # Stop if the API rate-limits or blocks requests.
        if error.code in [403, 429]:
            batch_reviews.append(review)
            print("Stopping this batch. Send me this output.")
            break

    except (URLError, TimeoutError, ValueError) as error:
        review["status"] = "request_failed"
        review["error"] = str(error)
        print("Request failed:", error)

    batch_reviews.append(review)

    # Pause briefly between requests.
    time.sleep(1)

# Save candidates and review status separately from the KSU data.
batch_path = interim_dir / f"crossref_sample_review_{batch_stamp}.json"
batch_path.write_text(
    json.dumps(batch_reviews, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nSaved review results:", batch_path.name)


PAPER 1
KSU title: Non-Linear Financial Intelligence Mixed-Integer Framework for Strategic Planning in Cost Management
KSU authors: Alkhanifer, A; Alamri, AM
KSU journal: PACIFIC BUSINESS REVIEW INTERNATIONAL

Candidate 1
Title: Optimizing Business Decision-Making Through Ai-Enhanced Business Intelligence Systems: A Systematic Review Of Data-Driven Insights In Financial And Strategic Planning
Authors: 
Journal: Strategic Data Management and Innovation
DOI: 10.71292/sdmi.v2i01.21
Published: [[2025, 1, 2]]
Abstract available: True

Candidate 2
Title: Linear and Mixed Integer Programming
Authors: Hartmut Stadtler
Journal: Supply Chain Management and Advanced Planning
DOI: 10.1007/3-540-24814-5_28
Published: None
Abstract available: False

Candidate 3
Title: Mixed Integer, Linear Programming Model for Multireservoir Strategic Planning
Authors: M.A. Saif; R. Kumar; M. Shanyoor
Journal: Proceedings of Middle East Oil Show
DOI: 10.2523/15759-ms
Published: [[1987, 3]]
Abstract available: Fals

In [28]:
#Papers 2, 3 and 4 have strong matches in Candidate 1: their titles, authors and journals align. 

# Paper 1 has no matching result among the three returned—leave its DOI unresolved and try OpenAlex later.

# These decisions apply only to the four papers we just reviewed.
expected_dois = {
    2: "10.1016/j.iot.2023.100986",
    3: "10.1109/tce.2023.3321846",
    4: "10.1109/tce.2024.3473291",
}

assert len(batch_reviews) == 4, "Expected four lookup results."

review_decisions = []

for number, result in enumerate(batch_reviews, start=1):
    decision = {
        "source_record": result["source_record"],
        "crossref_response_file": result.get("response_file"),
    }

    if number == 1:
        # No returned candidate matches this paper.
        decision.update({
            "match_status": "unresolved",
            "confirmed_doi": None,
            "reason": "None of the three returned titles match.",
            "next_action": "Search OpenAlex by title and verify authors.",
        })

    else:
        candidate = result["candidates"][0]
        doi = candidate.get("DOI", "").lower()

        # Ensure we are saving the candidate we actually reviewed.
        assert doi == expected_dois[number], "Unexpected DOI; stop."

        decision.update({
            "match_status": "reviewed_match",
            "confirmed_doi": doi,
            "reason": "Title, authors and journal align.",
            "crossref_published": candidate.get("published"),
            "abstract_available": bool(candidate.get("abstract")),
            "next_action": (
                "Look up abstract in OpenAlex using confirmed DOI."
                if "Abstract" in result["source_record"]["missing_fields"]
                else "Requested DOI found; retain existing KSU abstract."
            ),
        })

    review_decisions.append(decision)
    print(f"Paper {number}: {decision['match_status']}")

# Save separately from the original KSU datasets.
decisions_path = interim_dir / "ksu_four_paper_review_decisions.json"
decisions_path.write_text(
    json.dumps(review_decisions, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nSaved:", decisions_path.name)

Paper 1: unresolved
Paper 2: reviewed_match
Paper 3: reviewed_match
Paper 4: reviewed_match

Saved: ksu_four_paper_review_decisions.json


In [29]:
# get abstract for Paper 3 from OpenAlex using its confirmed DOI.


import json
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen

# Locate the project and load the saved review decisions.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

interim_dir = project_dir / "data" / "interim"

decisions = json.loads(
    (interim_dir / "ksu_four_paper_review_decisions.json")
    .read_text(encoding="utf-8")
)

# Select Paper 3 by the DOI we reviewed.
target_doi = "10.1109/tce.2023.3321846"
decision = next(
    item for item in decisions
    if item.get("confirmed_doi") == target_doi
)

# Request the exact DOI from OpenAlex.
lookup_url = (
    "https://" + "api.openalex.org/works/https://doi.org/"
    + target_doi
)

request = Request(
    lookup_url,
    headers={"User-Agent": "KSUResearchStudentProject/0.1"},
)

with urlopen(request, timeout=60) as response:
    response_bytes = response.read()

# Preserve the original API response.
api_dir = project_dir / "data" / "raw" / "openalex"
api_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
response_path = api_dir / f"paper3_{stamp}.json"
response_path.write_bytes(response_bytes)

work = json.loads(response_bytes)

# Confirm the returned DOI before using the abstract.
returned_doi = (work.get("doi") or "").lower()
assert returned_doi.removeprefix("https://doi.org/") == target_doi, (
    "DOI mismatch: stop and review."
)

# Reconstruct the abstract from OpenAlex's word positions.
abstract_index = work.get("abstract_inverted_index")
abstract = None

if abstract_index:
    positioned_words = [
        (position, word)
        for word, positions in abstract_index.items()
        for position in positions
    ]
    abstract = " ".join(
        word for position, word in sorted(positioned_words)
    )

# Save the lookup separately for review, including both source dates.
result = {
    "source_record": decision["source_record"],
    "confirmed_doi": target_doi,
    "openalex_id": work.get("id"),
    "openalex_title": work.get("display_name"),
    "abstract": abstract,
    "abstract_source": "OpenAlex" if abstract else None,
    "crossref_published": decision.get("crossref_published"),
    "openalex_publication_date": work.get("publication_date"),
    "response_file": response_path.relative_to(project_dir).as_posix(),
    "status": "pending_review",
}

(interim_dir / "ksu_paper3_openalex_review.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("KSU title:", decision["source_record"]["title"])
print("OpenAlex title:", work.get("display_name"))
print("DOI:", returned_doi)
print("Crossref date:", decision.get("crossref_published"))
print("OpenAlex date:", work.get("publication_date"))
print("\nAbstract:", abstract or "Not available")

KSU title: 6G-Enabled Consumer Electronics Device Intrusion Detection With Federated Meta-Learning and Digital Twins in a Meta-Verse Environment
OpenAlex title: 6G-Enabled Consumer Electronics Device Intrusion Detection With Federated Meta-Learning and Digital Twins in a Meta-Verse Environment
DOI: https://doi.org/10.1109/tce.2023.3321846
Crossref date: {'date-parts': [[2024, 2]]}
OpenAlex date: 2023-10-03

Abstract: The widespread adoption of consumer electronics devices coupled with the emergence of 6G technology has led to the establishment of an extensive network of interconnected devices, forming the underlying infrastructure of the Internet of Things (IoT). Nevertheless, this interconnectivity introduces a myriad of security concerns, given that these devices become susceptible to malicious activities and unauthorized breaches. Moreover, conventional intrusion detection systems encounter difficulties in managing imbalanced data scenarios, wherein the count of normal instances vas

In [30]:
# Paper 3 matches by title and DOI, 
# and OpenAlex Api supplied its abstract.

# Load Paper 3's saved lookup result.
paper3_path = interim_dir / "ksu_paper3_openalex_review.json"
paper3_review = json.loads(paper3_path.read_text(encoding="utf-8"))

# Confirm we are updating the intended paper.
assert paper3_review["confirmed_doi"] == "10.1109/tce.2023.3321846"

# Record the review outcome without changing the original KSU data.
paper3_review["status"] = "reviewed_match"
paper3_review["match_reason"] = (
    "OpenAlex title matches the KSU title, and its DOI matches "
    "the reviewed Crossref DOI."
)
paper3_review["quality_flags"] = [
    "publication_date_difference_needs_review"
]

paper3_path.write_text(
    json.dumps(paper3_review, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Paper 3: DOI and abstract reviewed.")
print("Both source dates preserved; date difference flagged.")

Paper 3: DOI and abstract reviewed.
Both source dates preserved; date difference flagged.


In [31]:
# Load the saved decisions and select Paper 4.
decisions = json.loads(
    (interim_dir / "ksu_four_paper_review_decisions.json")
    .read_text(encoding="utf-8")
)

target_doi = "10.1109/tce.2024.3473291"
decision4 = next(
    item for item in decisions
    if item.get("confirmed_doi") == target_doi
)

# Search OpenAlex using the reviewed DOI.
lookup_url = (
    "https://" + "api.openalex.org/works/https://doi.org/"
    + target_doi
)

request = Request(
    lookup_url,
    headers={"User-Agent": "KSUResearchStudentProject/0.1"},
)

with urlopen(request, timeout=60) as response:
    response_bytes = response.read()

# Preserve the original API response.
api_dir = project_dir / "data" / "raw" / "openalex"
api_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
response_path = api_dir / f"paper4_{stamp}.json"
response_path.write_bytes(response_bytes)

work4 = json.loads(response_bytes)

# Confirm that the returned DOI matches our request.
returned_doi = (work4.get("doi") or "").lower()
assert returned_doi.removeprefix("https://doi.org/") == target_doi, (
    "DOI mismatch: stop and review."
)

# Reconstruct the abstract when available.
abstract_index = work4.get("abstract_inverted_index")
abstract4 = None

if abstract_index:
    positioned_words = [
        (position, word)
        for word, positions in abstract_index.items()
        for position in positions
    ]
    abstract4 = " ".join(
        word for position, word in sorted(positioned_words)
    )

# Save this result separately for review.
paper4_review = {
    "source_record": decision4["source_record"],
    "confirmed_doi": target_doi,
    "openalex_id": work4.get("id"),
    "openalex_title": work4.get("display_name"),
    "abstract": abstract4,
    "abstract_source": "OpenAlex" if abstract4 else None,
    "crossref_published": decision4.get("crossref_published"),
    "openalex_publication_date": work4.get("publication_date"),
    "response_file": response_path.relative_to(project_dir).as_posix(),
    "status": "pending_review",
}

(interim_dir / "ksu_paper4_openalex_review.json").write_text(
    json.dumps(paper4_review, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("KSU title:", decision4["source_record"]["title"])
print("OpenAlex title:", work4.get("display_name"))
print("DOI:", returned_doi)
print("Crossref date:", decision4.get("crossref_published"))
print("OpenAlex date:", work4.get("publication_date"))
print("\nAbstract:", abstract4 or "Not available")

KSU title: 6GTelMED: Resources Recommendation Framework on 6G-Enabled Distributed Telemedicine Using Edge-AI
OpenAlex title: 6GTelMED: Resources Recommendation Framework on 6G-Enabled Distributed Telemedicine Using Edge-AI
DOI: https://doi.org/10.1109/tce.2024.3473291
Crossref date: {'date-parts': [[2024, 8]]}
OpenAlex date: 2024-08-01

Abstract: Telemedicine infrastructure is enhanced in recent times and applications developed have adopted base-line networking standards according to 4G/5G and LTE. The major challenge in exiting infrastructural setups is higher-latency and exposed privacy of resources and sensitive information. In this manuscript, we have proposed a 6G enabled resource recommendation framework for telemedicine. The framework is developed on the Edge-AI computational principles to cater the needs and demands of medical devices associated in telemedicine. The approach is to customize the network via Distributed Telemedicine Network (DTN) protocol for edge-devices such Io

In [32]:
# Paper 4 matches by title and DOI, and its abstract is available. 
# 
# Both sources indicate August 2024, 
# but only OpenAlex supplies a day. 

# Load Paper 4's saved lookup result.
paper4_path = interim_dir / "ksu_paper4_openalex_review.json"
paper4_review = json.loads(paper4_path.read_text(encoding="utf-8"))

# Check that we are updating the intended paper.
assert paper4_review["confirmed_doi"] == "10.1109/tce.2024.3473291"

# Record why we accepted the metadata match.
paper4_review["status"] = "reviewed_match"
paper4_review["match_reason"] = (
    "OpenAlex title matches the KSU title, and its DOI matches "
    "the reviewed Crossref DOI."
)

# Preserve the difference in date precision.
paper4_review["date_review"] = (
    "Both sources indicate August 2024. Crossref supplies year and "
    "month only; OpenAlex supplies 2024-08-01. The exact day has "
    "not been independently verified."
)
paper4_review["quality_flags"] = [
    "publication_date_precision_differs"
]

# Save separately from the original KSU records.
paper4_path.write_text(
    json.dumps(paper4_review, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Paper 4: DOI and abstract reviewed.")
print("Source dates and their precision preserved.")



Paper 4: DOI and abstract reviewed.
Source dates and their precision preserved.


In [35]:
from urllib.parse import urlencode

# Load the saved review decisions.
decisions = json.loads(
    (interim_dir / "ksu_four_paper_review_decisions.json")
    .read_text(encoding="utf-8")
)

# Identify the unresolved paper by its title.
paper1_title = (
    "Non-Linear Financial Intelligence Mixed-Integer Framework "
    "for Strategic Planning in Cost Management"
)


paper1 = next(
    item["source_record"]
    for item in decisions
    if item["source_record"]["title"] == paper1_title
)


# Search by title without assuming a publication year.
search_url = (
    "https://" + "api.openalex.org/works?"
    + urlencode({"search": paper1["title"], "per_page": 3})
)

request = Request(
    search_url,
    headers={"User-Agent": "KSUResearchStudentProject/0.1"},
)

with urlopen(request, timeout=60) as response:
    search_bytes = response.read()

# Preserve the original response.
api_dir = project_dir / "data" / "raw" / "openalex"
api_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
search_path = api_dir / f"paper1_search_{stamp}.json"
search_path.write_bytes(search_bytes)

search_data = json.loads(search_bytes)
paper1_candidates = search_data.get("results", [])

# Save the search details for later review.
search_note = {
    "source_record": paper1,
    "request_url": search_url,
    "response_file": search_path.relative_to(project_dir).as_posix(),
    "status": "pending_review" if paper1_candidates else "no_results",
}

(interim_dir / "ksu_paper1_openalex_search.json").write_text(
    json.dumps(search_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("KSU title:", paper1["title"])
print("KSU authors:", paper1["authors"])
print("KSU journal:", paper1["journal"])
print("\nTotal search results:", search_data.get("meta", {}).get("count"))


for number, candidate in enumerate(paper1_candidates, start=1):
    authors = "; ".join(
        (item.get("author") or {}).get("display_name", "")
        for item in candidate.get("authorships", [])
    )

    location = candidate.get("primary_location") or {}
    source = location.get("source") or {}

    print(f"\n--- Candidate {number} ---")
    print("Title:", candidate.get("display_name"))
    print("Authors:", authors)
    print("Journal:", source.get("display_name"))
    print("DOI:", candidate.get("doi"))
    print("Publication date:", candidate.get("publication_date"))
    print("OpenAlex ID:", candidate.get("id"))

if not paper1_candidates:
    print("\nNo results for this query. DOI remains unresolved.")

KSU title: Non-Linear Financial Intelligence Mixed-Integer Framework for Strategic Planning in Cost Management
KSU authors: Alkhanifer, A; Alamri, AM
KSU journal: PACIFIC BUSINESS REVIEW INTERNATIONAL

Total search results: 5009

--- Candidate 1 ---
Title: Big data analytics in logistics and supply chain management: Certain investigations for research and applications
Authors: Gang Wang; Angappa Gunasekaran; Eric W.T. Ngai; Θάνος Παπαδόπουλος
Journal: International Journal of Production Economics
DOI: https://doi.org/10.1016/j.ijpe.2016.03.014
Publication date: 2016-03-26
OpenAlex ID: https://openalex.org/W2302800291

--- Candidate 2 ---
Title: Supply chain risk management: a literature review
Authors: William Ho; Tian Zheng; Hakan Yildiz; Srinivas Talluri
Journal: International Journal of Production Research
DOI: https://doi.org/10.1080/00207543.2015.1030467
Publication date: 2015-04-15
OpenAlex ID: https://openalex.org/W2034717186

--- Candidate 3 ---
Title: Operating room planning a

In [34]:
#OpenAlex rate-limited the request. 
# This doesn’t mean the paper is missing. 


import sys
from urllib.error import HTTPError

# Read the previous error without contacting OpenAlex again.
last_error = getattr(sys, "last_value", None)

if isinstance(last_error, HTTPError):
    print("HTTP status:", last_error.code)
    print("Retry-After:", last_error.headers.get("Retry-After"))
    print(
        "Remaining budget:",
        last_error.headers.get("X-RateLimit-Remaining"),
    )
    print(
        "Reset information:",
        last_error.headers.get("X-RateLimit-Reset"),
    )
    print("\nServer explanation:")
    print(last_error.read().decode("utf-8", errors="replace")[:2000])
else:
    print("The previous error is no longer available.")


# output. It may tell us whether we need to wait or use an OpenAlex API key.


HTTP status: 429
Retry-After: 34
Remaining budget: None
Reset information: None

Server explanation:
{"error":"Rate limit exceeded","message":"Anonymous search is temporarily rate-limited while the search cluster is under elevated load. Please retry in 34s, or use a free API key for uninterrupted access: https://openalex.org/rest-api.","retryAfter":34}


In [36]:
# None of these three candidates matches Paper 1. Their titles, authors and journals differ. 
# The 5,009 results are search hits—not verified matches.

# For this small trial, leave Paper 1’s DOI unresolved. 
# That doesn’t prove it has no DOI; we simply haven’t found a reliable match.


# Load the saved OpenAlex search note.
paper1_note_path = interim_dir / "ksu_paper1_openalex_search.json"
paper1_note = json.loads(
    paper1_note_path.read_text(encoding="utf-8")
)

# Document that none of the inspected results matched.
paper1_note["status"] = "unresolved"
paper1_note["confirmed_doi"] = None
paper1_note["review_reason"] = (
    "None of the three Crossref candidates or three OpenAlex "
    "candidates inspected matched the KSU title, authors and journal."
)
paper1_note["next_action"] = (
    "Optional later review of the publisher website or a refined search."
)

paper1_note_path.write_text(
    json.dumps(paper1_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Paper 1: DOI remains unresolved.")
print("Original KSU record and existing abstract preserved.")

Paper 1: DOI remains unresolved.
Original KSU record and existing abstract preserved.


In [ ]:
import json
from pathlib import Path

# Locate the saved files, including after a Python restart.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

interim_dir = project_dir / "data" / "interim"

def load_saved(filename):
    return json.loads(
        (interim_dir / filename).read_text(encoding="utf-8")
    )

# Load the first successful sample and the four later decisions.
first_sample = load_saved("ksu_sample_enrichment.json")
decisions = load_saved("ksu_four_paper_review_decisions.json")
paper1_search = load_saved("ksu_paper1_openalex_search.json")
paper3 = load_saved("ksu_paper3_openalex_review.json")
paper4 = load_saved("ksu_paper4_openalex_review.json")

# Start with the first paper we successfully enriched.
summary_rows = [{
    "source_file_year": first_sample["source_file_year"],
    "source_row_index": first_sample["source_row_index"],
    "title": first_sample["source_title"],
    "match_status": "reviewed_match",
    "confirmed_doi": first_sample["confirmed_doi"],
    "abstract_retrieved": bool(first_sample.get("abstract")),
    "evidence_file": "ksu_sample_enrichment.json",
    "quality_flags": first_sample.get("quality_flags", []),
}]

# Add the four papers from the second batch.
for decision in decisions:
    source = decision["source_record"]
    doi = decision.get("confirmed_doi")

    row = {
        "source_file_year": source["source_file_year"],
        "source_row_index": source["source_row_index"],
        "title": source["title"],
        "match_status": decision["match_status"],
        "confirmed_doi": doi,
        "abstract_retrieved": False,
        "evidence_file": "ksu_four_paper_review_decisions.json",
        "quality_flags": [],
    }

    if not doi:
        row["match_status"] = paper1_search["status"]
        row["evidence_file"] = "ksu_paper1_openalex_search.json"

    elif doi == paper3["confirmed_doi"]:
        row["abstract_retrieved"] = bool(paper3.get("abstract"))
        row["evidence_file"] = "ksu_paper3_openalex_review.json"
        row["quality_flags"] = paper3.get("quality_flags", [])

    elif doi == paper4["confirmed_doi"]:
        row["abstract_retrieved"] = bool(paper4.get("abstract"))
        row["evidence_file"] = "ksu_paper4_openalex_review.json"
        row["quality_flags"] = paper4.get("quality_flags", [])

    summary_rows.append(row)

# Check that the summary contains five distinct source records.
source_keys = {
    (row["source_file_year"], row["source_row_index"])
    for row in summary_rows
}
assert len(summary_rows) == len(source_keys) == 5

# Save a summary; detailed metadata stays in the evidence files.
summary_path = interim_dir / "ksu_five_paper_trial_summary.json"
summary_path.write_text(
    json.dumps(summary_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Papers reviewed:", len(summary_rows))
print("Reviewed DOI matches:", sum(
    row["match_status"] == "reviewed_match" for row in summary_rows
))
print("Abstracts retrieved:", sum(
    row["abstract_retrieved"] for row in summary_rows
))
print("Unresolved papers:", sum(
    row["match_status"] == "unresolved" for row in summary_rows
))
print("Saved:", summary_path.name)




Papers reviewed: 5
Reviewed DOI matches: 4
Abstracts retrieved: 3
Unresolved papers: 1
Saved: ksu_five_paper_trial_summary.json


In [38]:
# Next: update the queue’s review status 
# so those five papers aren’t accidentally processed again. 

# We’ll save a new queue file, keeping the original unchanged.

# Load the original queue and completed trial summary.
queue = load_saved("ksu_enrichment_queue.json")
trial_results = load_saved("ksu_five_paper_trial_summary.json")

# Connect each trial result to its original source record.
results_by_key = {
    (item["source_file_year"], item["source_row_index"]): item
    for item in trial_results
}

updated_queue = []

for original_item in queue:
    item = original_item.copy()
    key = (item["source_file_year"], item["source_row_index"])

    if key in results_by_key:
        result = results_by_key[key]

        # These statuses describe lookup review, not a completed data merge.
        item["status"] = (
            "reviewed_match_pending_merge"
            if result["match_status"] == "reviewed_match"
            else "unresolved_after_trial"
        )
        item["review_summary_file"] = "ksu_five_paper_trial_summary.json"
        item["evidence_file"] = result["evidence_file"]

    updated_queue.append(item)

# Preserve the original queue and save the updated version separately.
updated_path = interim_dir / "ksu_enrichment_queue_reviewed.json"
updated_path.write_text(
    json.dumps(updated_queue, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Check that no records were lost.
assert len(updated_queue) == len(queue)

from collections import Counter

for status, count in Counter(
    item["status"] for item in updated_queue
).items():
    print(f"{status}: {count}")

print("Saved:", updated_path.name)


unresolved_after_trial: 1
reviewed_match_pending_merge: 4
pending: 2049
Saved: ksu_enrichment_queue_reviewed.json
